In [0]:
from pyspark.sql import SparkSession, DataFrame

def load_raw(config: dict, spark: SparkSession) -> DataFrame:
    source_type = config.get("source_type")
    conn = config.get("connection", {})

    if source_type == "cloud_storage":
        return cloud_files_load(
            spark,
            path=conn.get("cloud_storage_path"),
            file_format=conn.get("cloud_storage_type"),
            header=conn.get("header"),
            delimiter=conn.get("delimiter"),
            ingestion_mode=config.get("ingestion_mode", "snapshot"),
            schema_location=conn.get("cloud_schema_location"),
        )
    else:
        raise ValueError(f"Unsupported source_type: {source_type}")


def cloud_files_load(
    spark: SparkSession,
    *,
    path: str,
    file_format: str,
    ingestion_mode: str = "snapshot",
    header: bool = None,
    delimiter: str = None,
    schema_location: str = None,
) -> DataFrame:
    options = {"cloudFiles.format": file_format}
    if header is not None:
        options["header"] = str(header).lower()
    if delimiter:
        options["delimiter"] = delimiter
    if ingestion_mode == "streaming" and schema_location:
        options["cloudFiles.schemaLocation"] = schema_location

    reader = spark.readStream if ingestion_mode == "streaming" else spark.read
    return reader.format("cloudFiles").options(**options).load(path)